In [5]:
import tarfile
import gzip
import json
import os
import re
import pandas as pd
from pathlib import Path

archive_path = "data/newsroom-release.tar"
output_dir = "data/processed"
combined_output_path = os.path.join(output_dir, "solar_articles.jsonl")
split_output_paths = {
    "dev": os.path.join(output_dir, "solar_dev.jsonl"),
    "test": os.path.join(output_dir, "solar_test.jsonl"),
    "train": os.path.join(output_dir, "solar_train.jsonl"),
}

keywords = [
    "photovoltaic",
    "photovoltaics",
    "PV",
    "solar energy",
    "solar power",
    "solar panel",
    "solar panels",
    "solar farm",
    "solar thermal",
    "solar electricity",
    "solar generation",
    "rooftop solar",
    "utility-scale solar",
    "solar installation",
    "solar project",
    "solar capacity",
    "solar developer",
    "solar array",
    "solar module",
    "solar cell",
    "solar farms",
    "first solar",
    "solar cells",
    "solar industry",
    "solar company",
    "solar companies",
    "solar policy",
    "solar policies",
    "solar energies",
    "solar projects",
    "solar arrays",
    "solar installations",
    "residential solar",
    "solar technology",
    "solar powered",
    "solar plant",
    "solar plants",
    "solar market",
    "solar capacity",
    "solar electricity",
    "solar electric",
    "solar equipment",
    "solar sector",
    "solar roof"
]

keyword_re = re.compile(
    r"\b(?:%s)\b" % "|".join(re.escape(k) for k in keywords),
    re.IGNORECASE,
)

In [ ]:
def is_relevant(row):
    text = " ".join(
        str(row.get(k, "")) for k in row.keys() if isinstance(row.get(k), str)
    )
    return bool(keyword_re.search(text))


os.makedirs(output_dir, exist_ok=True)

with open(combined_output_path, "w", encoding="utf-8") as combined_f:
    for split in ["dev", "test", "train"]:
        split_total = 0
        split_matches = 0
        output_path = split_output_paths[split]

        with tarfile.open(archive_path, "r") as tf:
            member = None
            for candidate in tf.getmembers():
                if candidate.isreg() and candidate.name.endswith(f"{split}.jsonl.gz"):
                    member = candidate
                    break

            if member is None:
                print(f"No {split} split found")
                continue

            with tf.extractfile(member) as fh:
                if fh is None:
                    continue
                with gzip.GzipFile(fileobj=fh, mode="rb") as gz:
                    with open(output_path, "w", encoding="utf-8") as out_f:
                        for line in gz:
                            split_total += 1
                            line = line.decode("utf-8", errors="replace").strip()
                            if not line:
                                continue
                            try:
                                row = json.loads(line)
                            except json.JSONDecodeError:
                                continue

                            if is_relevant(row):
                                split_matches += 1
                                out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
                                combined_f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"{split}: {split_total} total rows, {split_matches} solar rows")

print(f"Saved split files to: {output_dir}")
print(f"Saved combined file to: {combined_output_path}")

# Optional count check after writing files
for path in split_output_paths.values():
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    print(f"{os.path.basename(path)}: {count}")


dev: 108837 total rows, 468 solar rows
test: 108862 total rows, 497 solar rows
train: 995041 total rows, 4437 solar rows
Saved split files to: data/processed
Saved combined file to: data/processed/solar_articles.jsonl
solar_dev.jsonl: 468
solar_test.jsonl: 497
solar_train.jsonl: 4437


In [9]:
input_path = Path("data/solar_articles_w_tokens.csv")
output_path = Path("data/solar_articles_w_tokens_filtered.csv")

def keyword_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return sum(1 for _ in keyword_re.finditer(text))


df = pd.read_csv(input_path)
df["keyword_hit_count"] = df["text"].fillna("").astype(str).apply(keyword_count)
filtered_df = df[df["keyword_hit_count"] >= 2].copy()
filtered_df.to_csv(output_path, index=False)

print(f"Input rows: {len(df)}")
print(f"Filtered rows: {len(filtered_df)}")
print(f"Saved filtered copy to: {output_path}")

Input rows: 5374
Filtered rows: 1985
Saved filtered copy to: data/solar_articles_w_tokens_filtered.csv
